<h2>Ejercicio 1</h2>

In [10]:
import pandas as pd
datos_orig = pd.read_csv("dataset.csv")
datos_orig

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13378,A-13379,Source2,2,2016-10-24 20:43:58,2016-10-24 21:13:39,38.694569,-122.886307,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
13379,A-13380,Source2,1,2016-10-24 20:54:32,2016-10-24 21:54:32,38.440739,-122.745216,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
13380,A-13381,Source2,3,2016-10-24 20:58:39,2016-10-24 21:58:17,37.955486,-122.053398,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
13381,A-13382,Source2,3,2016-10-24 20:52:38,2016-10-24 21:22:15,38.290943,-122.034760,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night


In [11]:
filtro = ['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 
'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)',
'Precipitation(in)', 'Weather_Condition','Sunrise_Sunset'] # filtro para datos necesarios

In [12]:
datos = datos_orig.dropna(subset=['Wind_Speed(mph)']) # no considerar los accidentes que no tengan registros de velocidad del viento 
datos = datos[filtro]
display(datos)

,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Sunrise_Sunset
2,36.0,33.3,100.0,29.67,10.0,SW,3.5,NaN,Overcast,Night
3,35.1,31.0,96.0,29.64,9.0,SW,4.6,NaN,Mostly Cloudy,Night
4,36.0,33.3,89.0,29.65,6.0,SW,3.5,NaN,Mostly Cloudy,Day
5,37.9,35.5,97.0,29.63,7.0,SSW,3.5,0.03,Light Rain,Day
6,34.0,31.0,100.0,29.66,7.0,WSW,3.5,NaN,Overcast,Day
...,...,...,...,...,...,...,...,...,...,...
13378,57.0,NaN,100.0,29.94,4.0,SSE,5.8,NaN,Rain,Night
13379,57.0,NaN,100.0,29.93,5.0,SSE,4.6,NaN,Rain,Night
13380,62.1,NaN,65.0,29.93,10.0,SSW,15.0,0.00,Overcast,Night
13381,62.1,NaN,74.0,29.97,10.0,SSW,8.1,0.00,Overcast,Night


In [13]:
import numpy as np
# Establecemos nuestra funcion con nuestros parametros para el indice de peligro
def peligro(df):
    puntaje = 0
    if df['Temperature(F)'] < 32:
        puntaje += 2
    elif df['Temperature(F)'] > 100:
        puntaje += 0.5
    if df['Wind_Chill(F)'] < 32:
        puntaje += 0.5
    if df['Humidity(%)'] > 90:
        puntaje += 0.5
    if df['Pressure(in)'] < 29.5:
        puntaje += 0.5
    if df['Visibility(mi)'] < 1:
        puntaje += 1.5
    elif 1 <= df['Visibility(mi)'] < 3:
        puntaje += 1
    elif 3 <= df['Visibility(mi)'] < 5:
        puntaje += 0.5
    if df['Wind_Direction'] == df['Wind_Direction']:
        puntaje += 0.5
    if df['Wind_Speed(mph)'] > 30:
        puntaje += 1
    elif 20 < df['Wind_Speed(mph)'] <= 30:
        puntaje += 0.5
    if df['Precipitation(in)'] > 0:
        # Crecimiento exponencial ajustado a la escala cuando exista precipitacion
        puntaje += np.exp(puntaje ** 0.05)
    if df['Weather_Condition'] in ['Light Freezing Drizzle','Light Freezing Rain',
    'Heavy Rain','Fog','Light Freezing Fog','Snow']:
        puntaje += 2
    elif df['Weather_Condition'] in ['Drizzle','Rain','Mist','Light Snow',
    'Light Thunderstorms and Rain','Light Rain Showers','Haze','Light Drizzle',
    'Patches of Fog','Light Rain']:
        puntaje += 1
    if df['Sunrise_Sunset'] == 'Night':
        puntaje += 1
    return puntaje / 2

datos['Peligro'] = datos.apply(peligro,axis=1) # aplicamos la funcion al df
datos['Peligro'] = datos['Peligro'].clip(lower=1).round() # nos aseguramos de que la escala tenga el 1 como minimo
df_peligro = pd.concat([datos_orig[['Start_Time','Start_Lat','Start_Lng','Description']],datos['Peligro']],axis=1).dropna()
df_peligro

,Start_Time,Start_Lat,Start_Lng,Description,Peligro
2,2016-02-08 06:49:27,39.063148,-84.032608,Accident on OH-32 State Route 32 Westbound at ...,1.0
3,2016-02-08 07:23:34,39.747753,-84.205582,Accident on I-75 Southbound at Exits 52 52B US...,1.0
4,2016-02-08 07:39:07,39.627781,-84.188354,Accident on McEwen Rd at OH-725 Miamisburg Cen...,1.0
5,2016-02-08 07:44:26,40.100590,-82.925194,Accident on I-270 Outerbelt Northbound near Ex...,2.0
6,2016-02-08 07:59:35,39.758274,-84.230507,Accident on Oakridge Dr at Woodward Ave. Expec...,1.0
...,...,...,...,...,...
13378,2016-10-24 20:43:58,38.694569,-122.886307,Right hand shoulder blocked due to accident on...,2.0
13379,2016-10-24 20:54:32,38.440739,-122.745216,Accident on Stony Point Rd Eastbound at Stony ...,2.0
13380,2016-10-24 20:58:39,37.955486,-122.053398,Right hand shoulder blocked due to accident on...,1.0
13381,2016-10-24 20:52:38,38.290943,-122.034760,Right hand shoulder blocked due to accident on...,1.0


<h2>Ejercicio 2</h2>

In [14]:
df_sign = pd.concat([datos_orig[['Traffic_Signal']], datos['Peligro']],axis=1).dropna()
# Dividimos en dos df para verificar los promedios de los indices por separado
df_True = df_sign[df_sign['Traffic_Signal'] == True]
df_False = df_sign[df_sign['Traffic_Signal'] == False]
display(df_True)
print(df_True['Peligro'].mean())
display(df_False)
print(df_False['Peligro'].mean())
x = df_False['Peligro'].mean() > df_True['Peligro'].mean() #booleana para corroborar
print(f'La afirmacion es: {x}')

,Traffic_Signal,Peligro
2,True,1.0
4,True,1.0
13,True,1.0
14,True,3.0
18,True,1.0
...,...,...
13327,True,1.0
13353,True,1.0
13356,True,1.0
13367,True,1.0


1.2211784799316823


,Traffic_Signal,Peligro
3,False,1.0
5,False,2.0
6,False,1.0
7,False,1.0
8,False,1.0
...,...,...
13378,False,2.0
13379,False,2.0
13380,False,1.0
13381,False,1.0


1.2440912795436023
La afirmacion es: True


De esta manera demostramos que la afirmacion *"los cruces sin señalización son más peligrosos que los cruces con señalización"* es correcta ya que la media del indice de peligro de aquellos cruces sin señalizacion es mayor que la de aquellos que no la tienen

<h2>Ejercicio 3</h2>

In [25]:
# Setear a datetime
datos_orig['Start_Time'] = pd.to_datetime(datos_orig['Start_Time'])
datos_orig['End_Time'] = pd.to_datetime(datos_orig['End_Time'])

# Calcular la diferencia de tiempo que nos dara la duracion
datos_orig['Duration'] = datos_orig['End_Time'] - datos_orig['Start_Time']
df_durac = datos_orig[['Description','Duration']].copy()

# Convertimos a horas
df_durac['hrs'] = round((df_durac['Duration'].dt.total_seconds() / 3600), 2)

# Aplicamos el cambio al indice original
df_durac['Peligro por duración'] = datos_orig.apply(peligro,axis=1).round()
df_durac['Peligro por duración'] += (df_durac['hrs']/24).round()
df_durac["Peligro por duración"]= df_durac["Peligro por duración"].clip(lower=1)
df_durac

,Description,Duration,hrs,Peligro por duración
0,Right lane blocked due to accident on I-70 Eas...,0 days 05:14:00,5.23,3.0
1,Accident on Brice Rd at Tussing Rd. Expect del...,0 days 00:30:00,0.50,2.0
2,Accident on OH-32 State Route 32 Westbound at ...,0 days 00:30:00,0.50,1.0
3,Accident on I-75 Southbound at Exits 52 52B US...,0 days 00:30:00,0.50,1.0
4,Accident on McEwen Rd at OH-725 Miamisburg Cen...,0 days 00:30:00,0.50,1.0
...,...,...,...,...
13378,Right hand shoulder blocked due to accident on...,0 days 00:29:41,0.49,2.0
13379,Accident on Stony Point Rd Eastbound at Stony ...,0 days 01:00:00,1.00,2.0
13380,Right hand shoulder blocked due to accident on...,0 days 00:59:38,0.99,1.0
13381,Right hand shoulder blocked due to accident on...,0 days 00:29:37,0.49,1.0


<h2>Ejercicio 4</h2>

<h3>A.-</h3>

In [16]:
df_ = datos_orig[datos_orig['Crossing']==True].copy()
df_["cruce"]= df_["Start_Lat"].fillna("").astype(str) + "-" + df_["End_Lat"].fillna("").astype(str) #convertimos a string y concatenamos las coordenadas
frec_cruce= df_["cruce"].value_counts().reset_index() # contamos los cruces y reseteamos los indices

frec_cruce.columns= ["Cruce (coordenadas)", "Frecuencia de accidentes"] #creamos las columnas 
frec_cruce.set_index("Cruce (coordenadas)", inplace=True)

display(frec_cruce)
print('Por lo tanto el cruce mas peligroso es:')
display(frec_cruce.head(1))

,Frecuencia de accidentes
Cruce (coordenadas),
37.66637-,18
37.878185-,12
38.452553-,11
37.170097-,11
37.90094000000001-,11
...,...
37.250286-,1
38.452599-,1
37.714367-,1


Por lo tanto el cruce mas peligroso es:


,Frecuencia de accidentes
Cruce (coordenadas),
37.66637-,18


<h3>B.-</h3>

In [17]:
# Dividimos en dos df para comparar promedios de peligro segun condiciones luminicas
df_luz = datos_orig[['Sunrise_Sunset']].copy()
df_luz['Peligro'] = datos_orig.apply(peligro,axis=1).round()
df_noche = df_luz[df_luz['Sunrise_Sunset']=='Night']
display(df_noche)
print(df_noche['Peligro'].mean())
df_dia = df_luz[df_luz['Sunrise_Sunset']=='Day']
display(df_dia)
print(df_dia['Peligro'].mean())

,Sunrise_Sunset,Peligro
0,Night,3.0
1,Night,2.0
2,Night,1.0
3,Night,1.0
32,Night,1.0
...,...,...
13378,Night,2.0
13379,Night,2.0
13380,Night,1.0
13381,Night,1.0


1.309620476610768


,Sunrise_Sunset,Peligro
4,Day,0.0
5,Day,2.0
6,Day,1.0
7,Day,1.0
8,Day,0.0
...,...,...
13352,Day,0.0
13353,Day,0.0
13354,Day,0.0
13355,Day,3.0


0.33506089660533817


Como podemos ver el promedio del indice de peligro de los accidentes ocurridos durante el dia es menor que el de los ocurridos durante la noche por lo cual es correcto afirmar que la condicion de luz mas peligrosa es la noche

<h3>C.-</h3>

In [18]:
# Dividimos en dos df para comparar promedios de peligro entre si es un cruce o no
df_cross = datos_orig[['Crossing']].copy()
df_cross['Peligro'] = datos_orig.apply(peligro,axis=1).round()
df_calles = df_cross[df_cross['Crossing']==False]
display(df_calles)
print(df_calles['Peligro'].mean())
df_cruces = df_cross[df_cross['Crossing']==True]
display(df_cruces)
print(df_cruces['Peligro'].mean())
inseg = df_calles['Peligro'].mean() > df_cruces['Peligro'].mean()
print(f'"Las calles son mas inseguras que los cruces": {inseg}')

,Crossing,Peligro
0,False,3.0
1,False,2.0
2,False,1.0
3,False,1.0
4,False,0.0
...,...,...
13378,False,2.0
13379,False,2.0
13380,False,1.0
13381,False,1.0


0.7544492741737062


,Crossing,Peligro
10,True,1.0
18,True,0.0
40,True,5.0
41,True,3.0
70,True,2.0
...,...,...
13305,True,0.0
13317,True,0.0
13347,True,0.0
13356,True,1.0


0.6773109243697479
"Las calles son mas inseguras que los cruces": True


Podemos afirmar que las calles son mas inseguras que los cruces